# Controlled CiteBench Comparison (Starter Implementation)

This notebook provides a first executable implementation for controlled (gold-context) CiteBench comparison between:
- Official RAGTruth baseline adapter output
- LettuceDetect adapter output

The notebook focuses on a working baseline with validation, error handling, and notebook-friendly unit tests.

## 1. Set Up Environment and Dependencies

In [ ]:
import importlib
import json
import os
import platform
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

required_modules = ["json", "pathlib", "subprocess"]
optional_modules = ["pandas", "yaml"]

for name in required_modules:
    importlib.import_module(name)
print("Required stdlib modules: OK")

for name in optional_modules:
    try:
        importlib.import_module(name)
        print(f"Optional module available: {name}")
    except Exception:
        print(f"Optional module missing: {name}")

## 2. Define Project Configuration

In [ ]:
@dataclass
class CompareConfig:
    project_root: Path
    prediction_jsonl: Path
    source_system_json: Path
    ragtruth_output_json: Path
    lettucedetect_input_json: Path
    context_source: str = "oracle"
    provider: str = "deepseek"
    model_name: str = "deepseek-chat"
    version: str = "citeeval-auto-12272024"
    modules: str = "ca,ce,cr_itercoe,cr_editdist"
    n_threads: int = 8
    max_samples: int = 30


PROJECT_ROOT = Path.cwd().resolve()
CONFIG = CompareConfig(
    project_root=PROJECT_ROOT,
    prediction_jsonl=PROJECT_ROOT / "benchmark" / "RAGTruth" / "baseline" / "prediction.jsonl",
    source_system_json=PROJECT_ROOT / "benchmark" / "CiteEval" / "data" / "system_eval" / "system_eval_examples.json",
    ragtruth_output_json=PROJECT_ROOT / "benchmark" / "CiteEval" / "data" / "system_eval" / "ragtruth_official.json",
    lettucedetect_input_json=PROJECT_ROOT / "benchmark" / "CiteEval" / "data" / "system_eval" / "lettucedetect.json",
)

CONFIG

## 3. Implement Core Functions

In [ ]:
def run_cmd(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess:
    """Run a command and print stdout/stderr for notebook visibility."""
    print("$", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


def path_exists_or_raise(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")


def convert_official_baseline(config: CompareConfig) -> Path:
    cmd = [
        sys.executable,
        "scripts/convert_ragtruth_baseline_to_citeeval.py",
        "--prediction-jsonl",
        str(config.prediction_jsonl),
        "--system-source",
        str(config.source_system_json),
        "--output",
        str(config.ragtruth_output_json),
        "--match-by",
        "query",
        "--allow-missing",
        "--max-samples",
        str(config.max_samples),
    ]
    run_cmd(cmd, cwd=config.project_root)
    path_exists_or_raise(config.ragtruth_output_json, "converted RAGTruth official JSON")
    return config.ragtruth_output_json


def run_controlled_compare(config: CompareConfig, output_dir: Path | None = None) -> Path:
    cmd = [
        sys.executable,
        "scripts/compare_citebench_methods.py",
        "--left-name",
        "ragtruth_official",
        "--left-input",
        str(config.ragtruth_output_json),
        "--right-name",
        "lettucedetect",
        "--right-input",
        str(config.lettucedetect_input_json),
        "--context-source",
        config.context_source,
        "--provider",
        config.provider,
        "--model-name",
        config.model_name,
        "--version",
        config.version,
        "--modules",
        config.modules,
        "--n-threads",
        str(config.n_threads),
        "--max-samples",
        str(config.max_samples),
    ]
    if output_dir is not None:
        cmd.extend(["--output-dir", str(output_dir)])

    run_cmd(cmd, cwd=config.project_root)

    if output_dir is None:
        base = config.project_root / "outputs" / "citebench_controlled_compare"
        candidates = sorted([p for p in base.glob("*") if p.is_dir()], reverse=True)
        if not candidates:
            raise FileNotFoundError(f"No comparison output directories found in {base}")
        return candidates[0]

    return output_dir


# Minimal inline example (not executed):
# converted_path = convert_official_baseline(CONFIG)
# run_dir = run_controlled_compare(CONFIG)

## 4. Add Input Validation and Error Handling

In [ ]:
def validate_config(config: CompareConfig) -> None:
    path_exists_or_raise(config.project_root, "project root")
    path_exists_or_raise(config.source_system_json, "CiteBench system source")

    allowed_context = {"oracle", "retrieval"}
    if config.context_source not in allowed_context:
        raise ValueError(f"context_source must be one of {allowed_context}")

    if config.max_samples <= 0:
        raise ValueError("max_samples must be > 0")


def validate_method_input(path: Path, label: str) -> None:
    path_exists_or_raise(path, label)
    rows = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"{label} must be a non-empty list JSON: {path}")
    required = {"id", "query", "passages", "pred"}
    missing = required - set(rows[0].keys())
    if missing:
        raise ValueError(f"{label} missing required keys in first row: {sorted(missing)}")


# Expected failure behavior checks
try:
    _bad = CompareConfig(
        project_root=Path("."),
        prediction_jsonl=Path("missing.jsonl"),
        source_system_json=CONFIG.source_system_json,
        ragtruth_output_json=Path("out.json"),
        lettucedetect_input_json=Path("lettuce.json"),
        max_samples=0,
    )
    validate_config(_bad)
except Exception as exc:
    print("Validation test OK (expected failure):", type(exc).__name__, str(exc)[:120])

## 5. Create Main Execution Flow

In [ ]:
# Set to False to execute conversions and evaluation commands.
DRY_RUN = True

validate_config(CONFIG)
print("Config validation: OK")

if DRY_RUN:
    print("DRY_RUN enabled: showing planned steps only.")
    print("1) Convert official baseline prediction JSONL -> CiteEval JSON")
    print("2) Validate both method input files")
    print("3) Run controlled method comparison and read summary")
else:
    converted = convert_official_baseline(CONFIG)
    print(f"Converted baseline file: {converted}")

    validate_method_input(CONFIG.ragtruth_output_json, "RAGTruth official adapted input")
    validate_method_input(CONFIG.lettucedetect_input_json, "LettuceDetect adapted input")

    output_dir = run_controlled_compare(CONFIG)
    print(f"Comparison output directory: {output_dir}")

    summary_path = output_dir / "summary.json"
    path_exists_or_raise(summary_path, "comparison summary")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2)[:4000])

## 6. Add Unit Tests and Run in VS Code

In [ ]:
import tempfile
import unittest


class NotebookFunctionTests(unittest.TestCase):
    def test_validate_config_rejects_bad_context(self):
        cfg = CompareConfig(
            project_root=Path("."),
            prediction_jsonl=Path("a.jsonl"),
            source_system_json=CONFIG.source_system_json,
            ragtruth_output_json=Path("b.json"),
            lettucedetect_input_json=Path("c.json"),
            context_source="invalid",
            max_samples=10,
        )
        with self.assertRaises(ValueError):
            validate_config(cfg)

    def test_validate_method_input_accepts_minimal_schema(self):
        with tempfile.TemporaryDirectory() as td:
            p = Path(td) / "sample.json"
            payload = [{"id": "1", "query": "q", "passages": [], "pred": "a [1]"}]
            p.write_text(json.dumps(payload), encoding="utf-8")
            validate_method_input(p, "test input")


suite = unittest.defaultTestLoader.loadTestsFromTestCase(NotebookFunctionTests)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)